# పాఠం 10 - ఉత్పత్తిలో AI ఏజెంట్లు

ఈ పాఠంలో మీరు Microsoft Agent Framework (Python) ఉపయోగించి AI ఏజెంట్ల కోసం **ఉత్పత్తి నమూనాలు** నేర్చుకుంటారు. మేము ఇవి చర్చిస్తాము:

- **పర్యవేక్షణ** — ఏజెంట్ పరస్పర చర్యలకు సమయనిర్ణయం మరియు లాగింగ్‌ను జోడించడం
- **మూల్యాంకనం** — ప్రతిస్పందన నాణ్యతను స్కోరు చేయడానికి ఒక మూల్యాంకక ఏజెంట్‌ను ఉపయోగించడం
- **ఖర్చు నిర్వహణ** — టోకెన్ ఆప్టిమైజేషన్ మరియు మోడల్ ఎంపికకు వ్యూహాలు

దృశ్యం ఒక **ప్రయాణ ఏజెంట్** ఇది వినియోగదారులకు ప్రయాణాలు పథకం చేసుకోవడంలో సహాయపడుతుంది, మరియు దీని మీద పర్యవేక్షణ మరియు మూల్యాంకనం అమలులో ఉన్నాయి.


## సెట్టప్


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
import time
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())


## ఉత్పత్తిలో పరిగణించాల్సిన అంశాలు

AI ఏజెంట్లను ప్రోటోటైప్స్ నుండి ప్రొడక్షన్‌కు మార్చేటప్పుడు మూడు స్తంభాలపై జాగ్రత్తగా దృష్టి పెట్టాలి:

1. **పర్యవేక్షణ** — ఏజెంట్ ఏమి చేస్తున్నదో, ఎంత సమయం పట్టతోందో, మరియు ఏ టూల్స్‌ను పిలుస్తుందో మీకు కనిపించాలి. ట్రేసింగ్ మరియు లాగింగ్ లేకపోతే, ప్రొడక్షన్ సమస్యలను డీబగ్గింగ్ చేయడం దాదాపు అసాధ్యం.

2. **మూల్యాంకనం** — ఆటోమేటెడ్ నాణ్యత తనిఖీలు ఏజెంట్ యొక్క స్పందనలు కాలానుగుణంగా ఖచ్చితంగా, పూర్తిగా, మరియు ఉపయోగకరంగా ఉన్నాయో నిర్ధారిస్తాయి. ఒక మూల్యాంకక ఏజెంట్ నిర్వచిత ప్రమాణాల మేరకు స్పందనలను స్కోర్ చేయగలదు.

3. **ఖర్చు నిర్వహణ** — టోకెన్ వినియోగం నేరుగా ఖర్చును ప్రభావితం చేస్తుంది. ప్రాంప్ట్ ఆప్టిమైజేషన్, మోడల్ ఎంపిక, మరియు క్యాషింగ్ వంటి వ్యూహాలు నాణ్యతను త్యాగం చేయకుండా ఖర్చులను నియంత్రణలో ఉంచడానికి సహాయపడతాయి.


## పర్యవేక్షించదగిన ఏజెంట్‌ను నిర్మించడం

మేము ప్రయాణ సంబంధి సాధనాలను నిర్వచించి, ఏజెంట్‌ను కాల్ చేసే చర్యకు సమయ కొలతను జోడిస్తాము, తద్వారా లేటెన్సీని పర్యవేక్షించగలుగుతాము. ప్రొడక్షన్‌లో మీరు OpenTelemetry లేదా సమానమైన ట్రేసింగ్ బ్యాకెండ్‌తో అనుసంధానిస్తారు.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## మూల్యాంకన నమూనాలు

సాధారణ ఉత్పత్తి నమూనాలో రెండవ ఏజెంట్‌ను **మూల్యాంకకుడు**గా ఉపయోగిస్తారు. మూల్యాంకకుడు ప్రాథమిక ఏజెంట్ యొక్క స్పందనను పూర్తితనం, ఖచ్చితత్వం, మరియు సహాయకత్వం వంటి ముందుగా నిర్వచించిన ప్రమాణాలపై స్కోర్ చేస్తుంది.

This enables:
- స్పందనలు వినియోగదారులకు చేరే ముందు ఆటోమేటెడ్ నాణ్యత గేట్లు
- ప్రాంప్ట్లు లేదా మోడళ్లు మారినప్పుడు రిగ్రెషన్ గుర్తింపు
- కాలంతో పాటు ఏజెంట్ పనితీరు యొక్క నిరంతర పర్యవేక్షణ


In [ ]:
evaluator = await provider.create_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## ఖర్చు నిర్వహణ వ్యూహాలు

ఉత్పత్తి AI ఏజెంట్లకు ఖర్చులను నియంత్రించడం అత్యవసరం. ఇక్కడ ముఖ్యమైన వ్యూహాలు ఉన్నాయి:

| వ్యూహం | వివరణ |
|---|---|
| **ప్రాంప్ట్ ఆప్టిమైజేషన్** | సిస్టమ్ సూచనలను సంక్షిప్తంగా ఉంచండి. ఇన్‌పుట్ టోకన్లను తగ్గించడానికి అవసరంలేని సందర్భాన్ని తొలగించండి. |
| **మోడల్ ఎంపిక** | వర్గీకరణ లేదా ఎక్స్‌ట్రాక్షన్ వంటి సాధారణ పనుల కోసం చిన్న, సస్తా మోడళ్లను (ఉదాహరణకు GPT-4o-mini) ఉపయోగించండి, మరియు సంక్లిష్ట తర్కానికి మాత్రమే పెద్ద మోడళ్లను కేటాయించండి. |
| **క్యాచింగ్** | టూల్ ఫలితాలను మరియు తరచుగా వచ్చే ప్రశ్నలను క్యాచ్ చేసి పునరావృత API కాల్‌లను నివారించండి. |
| **టోకెన్ బడ్జెట్లు** | అనూహ్యంగా దీర్ఘమైన ప్రతిస్పందనలను నివారించడానికి `max_tokens` పరిమితులను సెట్ చేయండి. |
| **బ్యాచ్ చేయడం** | సాధ్యమైన చోటు అనేక యూజర్ ప్రశ్నలను ఒకే API కాల్‌లో సమూహీకరించండి. |

వాస్తవంలో, స్థాయిగతమైన (tiered) 접근ం బాగా పనిచేస్తుంది: సులభమైన అభ్యర్థనలను వేగవంతమైన, తక్కువ ఖర్చు గల మోడల్‌కి పంపండి మరియు కేవలం సంక్లిష్ట ప్రశ్నలను మాత్రమే మరింత సామర్థ్యవంతమైన మోడల్‌కి ఎస్కలేట్ చేయండి.


## సారాంశం

ఈ పాఠంలో మీరు ఎలా చేయాలో నేర్చుకున్నారు:

1. **పరిశీలన సామర్థ్యాన్ని జోడించండి** ఏజెంట్ పరస్పర చర్యలకు కాలసమయ సమాచారం మరియు లాగింగ్ ద్వారా, ట్రేసింగ్ మరియు మానిటరింగ్ కోసం పునాది ఏర్పరచడం.
2. **ఏజెంట్ స్పందనలను అంచనా వేయండి** సమగ్రత, ఖచ్చితత్వం, మరియు సహాయకతకు స్కోరు ఇచ్చే ఎవాల్యువేటర్ ఏజెంట్ ఉపయోగించి స్వయంచాలకంగా.
3. **ఖర్చులను నిర్వహించండి** ప్రాంప్ట్ ఆప్టిమైజేషన్, మోడల్ ఎంపిక, క్యాషింగ్, మరియు టోకెన్ బడ్జెట్ల ద్వారా.

ఈ ఉత్పత్తి నమూనాలు మీ AI ఏజెంట్లు పెద్ద పరిమాణంలో నమ్మకమైనవి, కొలవగలవైనవి, మరియు ఖర్చు-సమర్థవంతంగా ఉండేలా నిర్ధారించడంలో సహాయపడతాయి.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
నిరాకరణ:
ఈ డాక్యుమెంట్‌ను AI అనువాద సేవ [Co-op Translator](https://github.com/Azure/co-op-translator) ద్వారా అనువదించారు. మేము ఖచ్చితత్వానికి యత్నించినప్పటికీ, స్వయంచాలక అనువాదాల్లో పొరపాట్లు లేదా తప్పిదాలు ఉండవచ్చు. మూల భాషలోని అసలైన డాక్యుమెంట్‌ను అధికారిక మూలంగా పరిగణించాలి. కీలకమైన సమాచారానికి వృత్తిపరమైన మానవ అనువాదాన్ని సిఫార్సు చేస్తాము. ఈ అనువాదాన్ని ఉపయోగించడం వల్ల కలిగే ఏవైనా అపార్థాలు లేదా తప్పు వ్యాఖ్యానాలకు మేము బాధ్యత వహించము.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
